# Spaceship Titanic | pipeline code 0.78185

- Pipeline are a sequence of data processing mechanisms. Pandas pipeline feature allows us to string together various user-defined Python functions in order to build a pipeline of data processing.
- It is used to chain multiple estimators into one and hence, automate the machine learning process. This is extremely useful as there are often a fixed sequence of steps in processing the data.

[<img src="https://i.ibb.co/WBf5YFL/screenshot-4.png" width=1000>](https://i.ibb.co/WBf5YFL/screenshot-4.png)

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

train = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

y = train['Transported']
X = train.drop('Transported', axis=1)
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size=0.7, test_size=0.3, random_state=0)

cat_cols = [cname for cname in X.columns if X[cname].nunique() < 10 and X[cname].dtype == "object"]
num_cols = [cname for cname in X.columns if X[cname].dtype in ['int64', 'float64']]

all_cols = cat_cols + num_cols
X_train = X_train_full[all_cols].copy()
X_valid = X_valid_full[all_cols].copy()
X_test = test[all_cols].copy()

print(f'CATEGORY: {cat_cols}\n')
print(f'NUMERIC: {num_cols}\n')

num_transform = SimpleImputer(strategy='constant')
cat_transform = Pipeline(steps = [ ('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore')) ])
preprocessor = ColumnTransformer(transformers = [ ('num', num_transform, num_cols), ('cat', cat_transform, cat_cols) ])
model = RandomForestRegressor(n_estimators=100, random_state=0)
clf = Pipeline(steps = [ ('preprocessor', preprocessor), ('model', model) ])

clf.fit(X_train, y_train)
preds = clf.predict(X_valid)
mae = mean_absolute_error(y_valid, preds)
print(f"MAE: {round(mae, 2)}\n")

answer = pd.read_csv('/kaggle/input/spaceship-titanic/sample_submission.csv')
answer['Transported'] = clf.predict(X_test)
answer['Transported'] = answer['Transported'].apply(lambda x: True if round(x) == 1 else False)
answer.to_csv('submission.csv', index=False)

CATEGORY: ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

NUMERIC: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

MAE: 0.28



![](https://i.ibb.co/fHJzctL/screenshot.png)

- `pipeline` it is very convenient for fast data processing and model building
- any comments are welcome
- thank you for your attention